# VisTAD: Baseline Students (1D-CNN & Res-TCN) × Dual Teacher Cache Manifolds

**Experimental Goal:**
Evaluates conventional time-series baselines (1D-CNN and Residual-TCN) across dual teacher distillation manifolds:
1. **New Cache (1024-d Ultra-VETime / DINOv2)**
2. **Old Cache (2048-d Early VLM)**

Evaluated across all 4 benchmarks: **SWaT, GECCO, SMD, MSL** with Range-Aware **Affiliation F1 (`Aff F1`)** and threshold-free **AUROC**.

In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

is_kaggle = os.path.exists('/kaggle')
repo_dir = '/kaggle/working/BaselineAblation' if is_kaggle else os.path.abspath('BaselineAblation')
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print(f'Working directory: {os.getcwd()}')
print('Unpacking self-contained Baseline Ablation codebase...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), 'r') as zf:
    zf.extractall('.')
print(f'Extracted top-level files: {sorted(os.listdir("."))}')


In [ ]:
# --- Step 2: Accelerator Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'Detected {n_gpus} CUDA GPU(s):')
    for i in range(n_gpus):
        prop = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {prop.name} | VRAM: {prop.total_memory / 1e9:.2f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('Apple Silicon MPS accelerator available.')
else:
    print('Running on CPU.')


In [ ]:
# --- Step 3: Multi-Dataset & Dual-Cache Auto-Discovery ---
import os, glob, shutil

for ds in ['swat', 'smd/test', 'smd/test_label', 'gecco', 'msl']:
    os.makedirs(os.path.join('data', ds), exist_ok=True)
os.makedirs('cache', exist_ok=True)
os.makedirs('checkpoints/baseline_ablation', exist_ok=True)
os.makedirs('results/ablations', exist_ok=True)

# Link Data Files from Kaggle or local parent directories
search_roots = ['/kaggle/input', '..', '.']
for root_dir in search_roots:
    if not os.path.exists(root_dir): continue
    # SWaT
    for p in glob.glob(os.path.join(root_dir, '**/SWaT_Dataset_*.csv'), recursive=True):
        target = os.path.join('data/swat', os.path.basename(p))
        if not os.path.exists(target): shutil.copy2(p, target)
    # SMD
    for p in glob.glob(os.path.join(root_dir, '**/smd/test/*.txt'), recursive=True):
        target = os.path.join('data/smd/test', os.path.basename(p))
        if not os.path.exists(target): shutil.copy2(p, target)
    for p in glob.glob(os.path.join(root_dir, '**/smd/test_label/*.txt'), recursive=True):
        target = os.path.join('data/smd/test_label', os.path.basename(p))
        if not os.path.exists(target): shutil.copy2(p, target)
    # GECCO
    for p in glob.glob(os.path.join(root_dir, '**/gecco_water.csv'), recursive=True):
        target = os.path.join('data/gecco', os.path.basename(p))
        if not os.path.exists(target): shutil.copy2(p, target)
    # MSL
    for p in glob.glob(os.path.join(root_dir, '**/msl_*.csv'), recursive=True):
        target = os.path.join('data/msl', os.path.basename(p))
        if not os.path.exists(target): shutil.copy2(p, target)
    # Teacher Caches
    for p in glob.glob(os.path.join(root_dir, '**/*.pt'), recursive=True):
        fname = os.path.basename(p)
        if 'cache' in fname.lower() or 'ultravetime' in fname.lower():
            target = os.path.join('cache', fname)
            if not os.path.exists(target):
                try: os.symlink(p, target)
                except OSError: shutil.copy2(p, target)

print(f'Linked data: SWaT={len(os.listdir("data/swat"))}, SMD={len(os.listdir("data/smd/test"))}, GECCO={len(os.listdir("data/gecco"))}, MSL={len(os.listdir("data/msl"))}')
print(f'Discovered caches in cache/: {sorted(os.listdir("cache"))}')


In [ ]:
# --- Step 4: Execute Baseline Cache Ablation Suite ---
import os, sys, subprocess, time
os.environ['PYTHONPATH'] = '.'

# Configure models to run: 'cnn_1d', 'res_tcn', or both
models_to_run = ['cnn_1d', 'res_tcn']  # or ['cnn_1d']
datasets_to_run = ['swat', 'gecco', 'smd', 'msl']

cmd = [
    sys.executable, 'run_baseline_cache_ablation.py',
    '--data_dir', 'data',
    '--cache_dir', 'cache',
    '--datasets'] + datasets_to_run + [
    '--models'] + models_to_run + [
    '--epochs', '5',
    '--lr', '0.0015',
    '--train_stride', '50',
    '--eval_stride', '20',
    '--num_workers', '2',
    '--batch_size', '32',
    '--checkpoint_dir', 'checkpoints/baseline_ablation',
    '--output_json', 'results/ablations/baseline_cache_ablation.json',
    '--output_report', 'results/ablations/BASELINE_CACHE_REPORT.md'
]

print(f'Executing command: {" ".join(cmd)}')
t0 = time.time()
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
process.stdout.close()
rc = process.wait()
print(f'Process exited with code: {rc} (Elapsed: {(time.time() - t0)/60:.2f} mins)')
if rc != 0:
    raise RuntimeError(f'Baseline ablation failed with code {rc}')


In [ ]:
# --- Step 5: Export Checkpoints & Reports to /kaggle/working/ ---
import os, shutil

if os.path.exists('/kaggle/working'):
    out_ckpt = '/kaggle/working/checkpoints/baseline_ablation'
    out_res = '/kaggle/working/results/ablations'
    os.makedirs(out_ckpt, exist_ok=True)
    os.makedirs(out_res, exist_ok=True)

    src_ck = 'checkpoints/baseline_ablation'
    if os.path.exists(src_ck):
        for f in os.listdir(src_ck):
            shutil.copy2(os.path.join(src_ck, f), os.path.join(out_ckpt, f))

    src_rep = 'results/ablations'
    if os.path.exists(src_rep):
        for f in os.listdir(src_rep):
            shutil.copy2(os.path.join(src_rep, f), os.path.join(out_res, f))
    print('Exported all checkpoints and reports to /kaggle/working/')


In [ ]:
# --- Step 6: Render Markdown Summary Table ---
import os
from IPython.display import display, Markdown

rep_path = 'results/ablations/BASELINE_CACHE_REPORT.md'
if os.path.exists(rep_path):
    with open(rep_path) as f:
        display(Markdown(f.read()))
else:
    print(f'Report not found at {rep_path}')
